In [ ]:
import torchvision.transforms as transforms
#from torchvision.datasets import MiniImagenet, EuroSAT
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18
from torchsummary import summary
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torchvision.datasets import ImageFolder
from PIL import Image
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, ConcatDataset
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset
from sklearn.model_selection import StratifiedShuffleSplit
import pickle
import os
from torch.utils.data import Dataset, random_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#!unzip "/content/drive/MyDrive/DL23_project/EuroSAT/EuroSAT.zip" -d "/content/drive/MyDrive/DL23_project/EuroSAT"
# !unzip "/content/drive/MyDrive/DeepLearning_Project/MiniImage/test.rar" -d "/content/drive/MyDrive/DeepLearning_Project/MiniImage"
# !unzip "/content/drive/MyDrive/DeepLearning_Project/MiniImage/train.rar" -d "/content/drive/MyDrive/DeepLearning_Project/MiniImage"
# !unzip "/content/drive/MyDrive/DeepLearning_Project/MiniImage/val.rar" -d "/content/drive/MyDrive/DeepLearning_Project/MiniImage"

# add GPU code
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# If you encounter some issues regarding cuda device, e.g., "RuntimeError: CUDA Out of memory error",
# try to switch the device to cpu by using the following code

# device = torch.device('cpu')

print('Device:', device)

In [ ]:
!tar -xvf "/content/drive/MyDrive/DL23/train.tar" -C "/content/drive/MyDrive/DL23"

In [ ]:


class MiniImageNet(ImageFolder):
    def __init__(self, root, split='train', transform=None, target_transform=None):
        self.root = root
        self.split = split
        self.transform = transform
        self.target_transform = target_transform
        self.loader = self._loader

        if split == 'train':
            self.root = root + '/train'
        elif split == 'val':
            self.root = root + '/val'
        elif split == 'test':
            self.root = root + '/test'
        else:
            raise ValueError("Invalid split. Use 'train', 'val', or 'test'.")

        super(MiniImageNet, self).__init__(self.root, transform=transform, target_transform=target_transform)

    def _loader(self, path):
        return Image.open(path).convert('RGB')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

miniimagenet_train = MiniImageNet(root='/content/drive/MyDrive/DL23',split='train', transform=transform)


num_train_samples = 400
num_val_samples = 100
num_test_samples = 100

classes = miniimagenet_train.classes

miniimagenet_train_split = []
miniimagenet_val_split = []
miniimagenet_test_split = []

for cls in classes:
    class_indices = [idx for idx, label in enumerate(miniimagenet_train.targets) if label == miniimagenet_train.class_to_idx[cls]]

    train_indices, test_indices = train_test_split(class_indices, test_size=num_test_samples, random_state=42)
    train_indices, val_indices = train_test_split(train_indices, test_size=num_val_samples, random_state=42)

    train_dataset = Subset(miniimagenet_train, train_indices)
    val_dataset = Subset(miniimagenet_train, val_indices)
    test_dataset = Subset(miniimagenet_train, test_indices)

    miniimagenet_train_split.append(train_dataset)
    miniimagenet_val_split.append(val_dataset)
    miniimagenet_test_split.append(test_dataset)

miniimagenet_train_final = ConcatDataset(miniimagenet_train_split)
miniimagenet_val_final = ConcatDataset(miniimagenet_val_split)
miniimagenet_test_final = ConcatDataset(miniimagenet_test_split)

miniimagenet_train_loader_final = DataLoader(miniimagenet_train_final, batch_size=32, shuffle=True)
miniimagenet_val_loader_final = DataLoader(miniimagenet_val_final, batch_size=32, shuffle=True)
miniimagenet_test_loader_final = DataLoader(miniimagenet_test_final, batch_size=32, shuffle=True)

In [ ]:
num_classes = len(miniimagenet_train.classes)
class CustomResNet18Extended(nn.Module):
    def __init__(self, num_classes):
        super(CustomResNet18Extended, self).__init__()
        resnet = resnet18(pretrained=True)
        self.features = nn.Sequential(
            *list(resnet.children())[:-1],  # Remove the last fully connected layer
            # nn.Conv2d(512, 256, kernel_size=3, stride=1, padding=1),
            # nn.BatchNorm2d(256),
            # nn.ReLU(inplace=True),

            # nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1),
            # nn.BatchNorm2d(128),
            # nn.ReLU(inplace=True),

            # nn.Conv2d(128, 64, kernel_size=3, stride=1, padding=1),
            # nn.BatchNorm2d(64),
            # nn.ReLU(inplace=True),
        )
        # self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # x = self.features(x)
        # x = F.adaptive_avg_pool2d(x, (1, 1))
        # x = x.view(x.size(0), -1)
        # # x = self.dropout(x)
        # x = self.fc(x)
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
model_extended = CustomResNet18Extended(num_classes).to(device)
#summary(model_extended, (3, 224, 224))
criterion_extended = nn.CrossEntropyLoss()
optimizer_extended = optim.SGD(model_extended.parameters(), lr=0.001, momentum=0.9)


In [ ]:
def validate(model, val_loader, criterion):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = correct / total
    average_loss = val_loss / len(val_loader)

    return average_loss, accuracy

def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = correct / total
    average_loss = test_loss / len(test_loader)

    return average_loss, accuracy


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0, path='checkpoint.pth'):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        print(f'Validation loss decreased. Saving model ...')
        torch.save(model.state_dict(), self.path)

In [ ]:
def train_extended(model, train_loader, val_loader, test_loader, criterion, optimizer, early_stopping, num_epochs=15):
    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        average_train_loss = running_loss / len(train_loader)
        train_losses.append(average_train_loss)

        # Validation
        val_loss, val_accuracy = validate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch {epoch + 1}/{num_epochs}: "
              f"Train Loss: {average_train_loss:.4f}, "
              f"Validation Loss: {val_loss:.4f}, "
              f"Validation Accuracy: {val_accuracy:.4f}")
        early_stopping(val_loss, model)

        if early_stopping.early_stop:
            print("Early stopping")
            break

    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    # Test
    test_loss, test_accuracy = test(model, test_loader, criterion)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


In [ ]:
early_stopping = EarlyStopping(patience=5, delta=0.001)
train_extended(model_extended, miniimagenet_train_loader_final, miniimagenet_val_loader_final, miniimagenet_test_loader_final, criterion_extended, optimizer_extended,early_stopping)


In [ ]:
torch.save(model_extended.state_dict(), '/content/drive/MyDrive/DL23/pretrained_model_miniimagenet_extended.pth')

### **Part two:**

In [ ]:
!unzip "/content/drive/MyDrive/DL23/EuroSAT.zip" -d "/content/drive/MyDrive/DL23"


In [ ]:
eurosat_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
eurosat_dataset = ImageFolder(root='/content/drive/MyDrive/DL23/2750', transform=eurosat_transform)


In [ ]:
classes = eurosat_dataset.classes
classes

In [ ]:
selected_classes = np.random.choice(classes, 5, replace=False)
selected_classes

In [ ]:
selected_images = []
images_per_class = 20

In [ ]:
dataset_root = '/content/drive/MyDrive/DL23/2750'
for class_name in selected_classes:
    class_path = os.path.join(dataset_root, class_name)

    all_images = [img for img in os.listdir(class_path) if img.endswith('.jpg')]

    selected_images.extend([(class_name, img) for img in np.random.choice(all_images, images_per_class, replace=False)])


class CustomDataset(Dataset):
    def __init__(self, root, image_list, transform=None):
        self.root = root
        self.image_list = image_list
        self.transform = transform

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        class_name, img_name = self.image_list[idx]
        img_path = os.path.join(self.root, class_name, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, class_name


In [ ]:
selected_dataset = CustomDataset(root=dataset_root, image_list=selected_images, transform=eurosat_transform)


In [ ]:
class_count = defaultdict(int)

for _, class_name in selected_dataset:
    class_count[class_name] += 1

print(f"Number of classes: {len(class_count)}")
print("Number of images per class:")
for class_name, count in class_count.items():
    print(f"{class_name}: {count} images")

In [ ]:
selected_dataset_path = '/content/drive/MyDrive/DL23/selected_dataset.pkl'

with open(selected_dataset_path, 'wb') as file:
    pickle.dump(selected_dataset, file)

print(f"Selected dataset saved to: {selected_dataset_path}")

In [ ]:
with open(selected_dataset_path, 'rb') as file:
    selected_dataset1 = pickle.load(file)

In [ ]:
class_count_loaded = defaultdict(int)
total_images_loaded = 0

for _, class_name in selected_dataset1:
    class_count_loaded[class_name] += 1
    total_images_loaded += 1

print("Loaded Selected Dataset:")
print(f"Number of classes: {len(class_count_loaded)}")
print(f"Total number of images: {total_images_loaded}")
print("Number of images per class:")
for class_name, count in class_count_loaded.items():
    print(f"{class_name}: {count} images")


In [ ]:
features = [item[0] for item in selected_dataset1]
labels = [item[1] for item in selected_dataset1]

stratified_splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.75, random_state=42)

for train_index, test_index in stratified_splitter.split(features, labels):
    train_data = [(features[i], labels[i]) for i in train_index]
    test_data = [(features[i], labels[i]) for i in test_index]

print("The length of train data is:", len(train_data))
print("The length of test data is:", len(test_data))


In [ ]:

label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform([item[1] for item in train_data])
test_labels_encoded = label_encoder.transform([item[1] for item in test_data])

train_features_tensor = torch.stack([item[0] for item in train_data])
train_labels_tensor = torch.tensor(train_labels_encoded, dtype=torch.long)

test_features_tensor = torch.stack([item[0] for item in test_data])
test_labels_tensor = torch.tensor(test_labels_encoded, dtype=torch.long)

train_dataset = TensorDataset(train_features_tensor, train_labels_tensor)
test_dataset = TensorDataset(test_features_tensor, test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=25, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=len(test_data), shuffle=False)

In [ ]:

class_count = defaultdict(int)

for _, label in train_data:
    class_count[label] += 1

for class_label, count in class_count.items():
    print(f"Class {class_label}: {count} pictures in the training data")


In [ ]:
class_count = defaultdict(int)

for _, label in test_data:
    class_count[label] += 1

for class_label, count in class_count.items():
    print(f"Class {class_label}: {count} pictures in the test data")


In [ ]:
pretrained_model_path = '/content/drive/MyDrive/DL23/pretrained_model_miniimagenet_extended.pth'
model_schmodel = CustomResNet18Extended(num_classes=64)
model_schmodel.load_state_dict(torch.load(pretrained_model_path, map_location=torch.device('cpu')))
model_schmodel.eval()
print(model_schmodel)
num_classes_new_dataset = 5
model_schmodel.fc = nn.Linear(model_schmodel.fc.in_features, num_classes_new_dataset)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_schmodel.parameters(), lr=0.001)

In [ ]:
training_losses = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        optimizer.zero_grad()

        outputs = model_schmodel(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    training_losses.append(epoch_loss)


    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss}") # avg training loss for this epoch

plt.plot(range(1, num_epochs + 1), training_losses, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
# Testing
model_schmodel.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model_schmodel(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%") # print the accuracy on the test set

In [ ]:
torch.save(model_schmodel.state_dict(), 'model_schmodel.pth')

In [ ]:
selected_classes2 = np.random.choice(classes, 5, replace=False)
selected_classes2
selected_images2 = []

for class_name in selected_classes2:
    class_path = os.path.join(dataset_root, class_name)

    all_images = [img for img in os.listdir(class_path) if img.endswith('.jpg')]

    selected_images2.extend([(class_name, img) for img in np.random.choice(all_images, images_per_class, replace=False)])

selected_dataset2 = CustomDataset(root=dataset_root, image_list=selected_images2, transform=eurosat_transform)
class_count = defaultdict(int)

for _, class_name in selected_dataset2:
    class_count[class_name] += 1

print(f"Number of classes: {len(class_count)}")
print("Number of images per class:")
for class_name, count in class_count.items():
    print(f"{class_name}: {count} images")



selected_dataset2_path = '/content/drive/MyDrive/DL23/selected_dataset2.pkl'

with open(selected_dataset2_path, 'wb') as file:
    pickle.dump(selected_dataset2, file)

print(f"Selected dataset saved to: {selected_dataset2_path}")

with open(selected_dataset2_path, 'rb') as file:
    selected_dataset2 = pickle.load(file)


features = [item[0] for item in selected_dataset2]
labels = [item[1] for item in selected_dataset2]

stratified_splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.75, random_state=42)

for train_index, test_index in stratified_splitter.split(features, labels):
    train_data2 = [(features[i], labels[i]) for i in train_index]
    test_data2 = [(features[i], labels[i]) for i in test_index]

print("The length of train data is:", len(train_data2))
print("The length of test data is:", len(test_data2))


label_encoder = LabelEncoder()
train_labels_encoded2 = label_encoder.fit_transform([item[1] for item in train_data2])
test_labels_encoded2 = label_encoder.transform([item[1] for item in test_data2])

train_features_tensor2 = torch.stack([item[0] for item in train_data2])
train_labels_tensor2 = torch.tensor(train_labels_encoded2, dtype=torch.long)

test_features_tensor2 = torch.stack([item[0] for item in test_data2])
test_labels_tensor2 = torch.tensor(test_labels_encoded2, dtype=torch.long)

train_dataset2 = TensorDataset(train_features_tensor2, train_labels_tensor2)
test_dataset2 = TensorDataset(test_features_tensor2, test_labels_tensor2)

train_loader2 = DataLoader(train_dataset2, batch_size=25, shuffle=True)
test_loader2 = DataLoader(test_dataset2, batch_size=len(test_data2), shuffle=False)


In [ ]:
pretrained_model_path = '/content/drive/MyDrive/DL23/pretrained_model_miniimagenet_extended.pth'
model_schmodel2 = CustomResNet18Extended(num_classes=64)
model_schmodel2.load_state_dict(torch.load(pretrained_model_path, map_location=torch.device('cpu')))
model_schmodel2.eval()
print(model_schmodel2)
num_classes_new_dataset = 5
model_schmodel2.fc = nn.Linear(model_schmodel2.fc.in_features, num_classes_new_dataset)

criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(model_schmodel2.parameters(), lr=0.01)

training_losses2 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel2.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader2):
        optimizer.zero_grad()

        outputs2 = model_schmodel2(inputs)
        loss = criterion2(outputs2, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss2 = running_loss / len(train_loader2)
    training_losses2.append(epoch_loss2)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss2}")

plt.plot(range(1, num_epochs + 1), training_losses2, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()


model_schmodel2.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader2:
        outputs2 = model_schmodel2(inputs)
        _, predicted = torch.max(outputs2.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy2 = correct / total
print(f"Test Accuracy: {accuracy2 * 100:.2f}%")

torch.save(model_schmodel2.state_dict(), 'model_schmodel2.pth')

In [ ]:
selected_classes3 = np.random.choice(classes, 5, replace=False)
selected_classes3
selected_images3 = []

for class_name in selected_classes3:
    class_path = os.path.join(dataset_root, class_name)

    all_images = [img for img in os.listdir(class_path) if img.endswith('.jpg')]

    selected_images3.extend([(class_name, img) for img in np.random.choice(all_images, images_per_class, replace=False)])

selected_dataset3 = CustomDataset(root=dataset_root, image_list=selected_images3, transform=eurosat_transform)
class_count = defaultdict(int)

for _, class_name in selected_dataset3:
    class_count[class_name] += 1

print(f"Number of classes: {len(class_count)}")
print("Number of images per class:")
for class_name, count in class_count.items():
    print(f"{class_name}: {count} images")



selected_dataset3_path = '/content/drive/MyDrive/DL23/selected_dataset3.pkl'

with open(selected_dataset3_path, 'wb') as file:
    pickle.dump(selected_dataset3, file)

print(f"Selected dataset saved to: {selected_dataset3_path}")

with open(selected_dataset3_path, 'rb') as file:
    selected_dataset3 = pickle.load(file)


features = [item[0] for item in selected_dataset3]
labels = [item[1] for item in selected_dataset3]

stratified_splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.75, random_state=42)

for train_index, test_index in stratified_splitter.split(features, labels):
    train_data3 = [(features[i], labels[i]) for i in train_index]
    test_data3 = [(features[i], labels[i]) for i in test_index]

print("The length of train data is:", len(train_data3))
print("The length of test data is:", len(test_data3))


label_encoder = LabelEncoder()
train_labels_encoded3 = label_encoder.fit_transform([item[1] for item in train_data3])
test_labels_encoded3 = label_encoder.transform([item[1] for item in test_data3])

train_features_tensor3 = torch.stack([item[0] for item in train_data3])
train_labels_tensor3 = torch.tensor(train_labels_encoded3, dtype=torch.long)

test_features_tensor3 = torch.stack([item[0] for item in test_data3])
test_labels_tensor3 = torch.tensor(test_labels_encoded3, dtype=torch.long)

train_dataset3 = TensorDataset(train_features_tensor3, train_labels_tensor3)
test_dataset3 = TensorDataset(test_features_tensor3, test_labels_tensor3)

train_loader3 = DataLoader(train_dataset3, batch_size=25, shuffle=True)
test_loader3 = DataLoader(test_dataset3, batch_size=len(test_data3), shuffle=False)


In [ ]:
pretrained_model_path = '/content/drive/MyDrive/DL23/pretrained_model_miniimagenet_extended.pth'
model_schmodel3 = CustomResNet18Extended(num_classes=64)
model_schmodel3.load_state_dict(torch.load(pretrained_model_path, map_location=torch.device('cpu')))
model_schmodel3.eval()
print(model_schmodel3)
num_classes_new_dataset = 5
model_schmodel3.fc = nn.Linear(model_schmodel3.fc.in_features, num_classes_new_dataset)

criterion3 = nn.CrossEntropyLoss()
optimizer3 = optim.SGD(model_schmodel3.parameters(), lr=0.001, momentum=0.9)

training_losses3 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel3.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader3):
        optimizer.zero_grad()

        outputs3 = model_schmodel3(inputs)
        loss = criterion3(outputs3, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss3 = running_loss / len(train_loader3)
    training_losses3.append(epoch_loss3)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss3}")


plt.plot(range(1, num_epochs + 1), training_losses3, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()


model_schmodel3.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader3:
        outputs3 = model_schmodel3(inputs)
        _, predicted = torch.max(outputs3.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy3 = correct / total
print(f"Test Accuracy: {accuracy3 * 100:.2f}%")

torch.save(model_schmodel3.state_dict(), 'model_schmodel3.pth')

In [ ]:
selected_classes4 = np.random.choice(classes, 5, replace=False)
selected_classes4
selected_images4 = []

for class_name in selected_classes4:
    class_path = os.path.join(dataset_root, class_name)

    all_images = [img for img in os.listdir(class_path) if img.endswith('.jpg')]
    selected_images4.extend([(class_name, img) for img in np.random.choice(all_images, images_per_class, replace=False)])

selected_dataset4 = CustomDataset(root=dataset_root, image_list=selected_images4, transform=eurosat_transform)
class_count = defaultdict(int)

for _, class_name in selected_dataset4:
    class_count[class_name] += 1

print(f"Number of classes: {len(class_count)}")
print("Number of images per class:")
for class_name, count in class_count.items():
    print(f"{class_name}: {count} images")



selected_dataset4_path = '/content/drive/MyDrive/DL23/selected_dataset4.pkl'

with open(selected_dataset4_path, 'wb') as file:
    pickle.dump(selected_dataset4, file)

print(f"Selected dataset saved to: {selected_dataset4_path}")

with open(selected_dataset4_path, 'rb') as file:
    selected_dataset4 = pickle.load(file)

features = [item[0] for item in selected_dataset4]
labels = [item[1] for item in selected_dataset4]

stratified_splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.75, random_state=42)

for train_index, test_index in stratified_splitter.split(features, labels):
    train_data4 = [(features[i], labels[i]) for i in train_index]
    test_data4 = [(features[i], labels[i]) for i in test_index]

print("The length of train data is:", len(train_data4))
print("The length of test data is:", len(test_data4))


label_encoder = LabelEncoder()
train_labels_encoded4 = label_encoder.fit_transform([item[1] for item in train_data4])
test_labels_encoded4 = label_encoder.transform([item[1] for item in test_data4])

train_features_tensor4 = torch.stack([item[0] for item in train_data4])
train_labels_tensor4 = torch.tensor(train_labels_encoded4, dtype=torch.long)

test_features_tensor4 = torch.stack([item[0] for item in test_data4])
test_labels_tensor4 = torch.tensor(test_labels_encoded4, dtype=torch.long)

train_dataset4 = TensorDataset(train_features_tensor4, train_labels_tensor4)
test_dataset4 = TensorDataset(test_features_tensor4, test_labels_tensor4)

train_loader4 = DataLoader(train_dataset4, batch_size=25, shuffle=True)
test_loader4 = DataLoader(test_dataset4, batch_size=len(test_data4), shuffle=False)


In [ ]:
pretrained_model_path = '/content/drive/MyDrive/DL23/pretrained_model_miniimagenet_extended.pth'
model_schmodel4 = CustomResNet18Extended(num_classes=64)
model_schmodel4.load_state_dict(torch.load(pretrained_model_path, map_location=torch.device('cpu')))
model_schmodel4.eval()
print(model_schmodel4)
num_classes_new_dataset = 5
model_schmodel4.fc = nn.Linear(model_schmodel4.fc.in_features, num_classes_new_dataset)

criterion4 = nn.CrossEntropyLoss()
optimizer4 = optim.SGD(model_schmodel4.parameters(), lr=0.001, momentum=0.5)

training_losses4 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel4.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader4):
        optimizer.zero_grad()

        outputs4 = model_schmodel4(inputs)
        loss = criterion4(outputs4, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    epoch_loss4 = running_loss / len(train_loader4)
    training_losses4.append(epoch_loss4)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss4}")

plt.plot(range(1, num_epochs + 1), training_losses4, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

model_schmodel4.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader4:
        outputs4 = model_schmodel4(inputs)
        _, predicted = torch.max(outputs4.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy4 = correct / total
print(f"Test Accuracy: {accuracy4 * 100:.2f}%")

torch.save(model_schmodel4.state_dict(), 'model_schmodel4.pth')

In [ ]:
selected_classes5 = np.random.choice(classes, 5, replace=False)
selected_classes5
selected_images5 = []

for class_name in selected_classes5:
    class_path = os.path.join(dataset_root, class_name)

    all_images = [img for img in os.listdir(class_path) if img.endswith('.jpg')]
    selected_images5.extend([(class_name, img) for img in np.random.choice(all_images, images_per_class, replace=False)])

selected_dataset5 = CustomDataset(root=dataset_root, image_list=selected_images5, transform=eurosat_transform)
class_count = defaultdict(int)

for _, class_name in selected_dataset5:
    class_count[class_name] += 1

print(f"Number of classes: {len(class_count)}")
print("Number of images per class:")
for class_name, count in class_count.items():
    print(f"{class_name}: {count} images")


selected_dataset5_path = '/content/drive/MyDrive/DL23/selected_dataset5.pkl'

with open(selected_dataset5_path, 'wb') as file:
    pickle.dump(selected_dataset5, file)

print(f"Selected dataset saved to: {selected_dataset5_path}")

with open(selected_dataset5_path, 'rb') as file:
    selected_dataset5 = pickle.load(file)

features = [item[0] for item in selected_dataset5]
labels = [item[1] for item in selected_dataset5]

stratified_splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.75, random_state=42)

for train_index, test_index in stratified_splitter.split(features, labels):
    train_data5 = [(features[i], labels[i]) for i in train_index]
    test_data5 = [(features[i], labels[i]) for i in test_index]

print("The length of train data is:", len(train_data5))
print("The length of test data is:", len(test_data5))

label_encoder = LabelEncoder()
train_labels_encoded5 = label_encoder.fit_transform([item[1] for item in train_data5])
test_labels_encoded5 = label_encoder.transform([item[1] for item in test_data5])

train_features_tensor5 = torch.stack([item[0] for item in train_data5])
train_labels_tensor5 = torch.tensor(train_labels_encoded5, dtype=torch.long)

test_features_tensor5 = torch.stack([item[0] for item in test_data5])
test_labels_tensor5 = torch.tensor(test_labels_encoded5, dtype=torch.long)

train_dataset5 = TensorDataset(train_features_tensor5, train_labels_tensor5)
test_dataset5 = TensorDataset(test_features_tensor5, test_labels_tensor5)

train_loader5 = DataLoader(train_dataset5, batch_size=25, shuffle=True)
test_loader5 = DataLoader(test_dataset5, batch_size=len(test_data5), shuffle=False)


In [ ]:
pretrained_model_path = '/content/drive/MyDrive/DL23/pretrained_model_miniimagenet_extended.pth'
model_schmodel5 = CustomResNet18Extended(num_classes=64)
model_schmodel5.load_state_dict(torch.load(pretrained_model_path, map_location=torch.device('cpu')))
model_schmodel5.eval()
print(model_schmodel5)
num_classes_new_dataset = 5
model_schmodel5.fc = nn.Linear(model_schmodel5.fc.in_features, num_classes_new_dataset)

criterion5 = nn.CrossEntropyLoss()
optimizer5 = optim.SGD(model_schmodel5.parameters(), lr=0.1, momentum=0.9)

training_losses5 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel5.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader5):
        optimizer.zero_grad()

        outputs5 = model_schmodel5(inputs)
        loss = criterion5(outputs5, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss5 = running_loss / len(train_loader5)
    training_losses5.append(epoch_loss5)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss5}")


plt.plot(range(1, num_epochs + 1), training_losses5, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()


# Testing
model_schmodel5.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader5:
        outputs5 = model_schmodel5(inputs)
        _, predicted = torch.max(outputs5.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy5 = correct / total
print(f"Test Accuracy: {accuracy5 * 100:.2f}%")

torch.save(model_schmodel5.state_dict(), 'model_schmodel5.pth')

In [ ]:
plt.plot(range(1, num_epochs + 1), training_losses, label = "Loss for model_schmodel")
plt.plot(range(1, num_epochs + 1), training_losses2, label = "Loss for model_schmodel2")
plt.plot(range(1, num_epochs + 1), training_losses3, label = "Loss for model_schmodel3")
plt.plot(range(1, num_epochs + 1), training_losses4, label = "Loss for model_schmode4")
plt.plot(range(1, num_epochs + 1), training_losses5, label = "Loss for model_schmode5")
plt.legend()
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

print(f"Average test accuracy: {(accuracy + accuracy2 + accuracy3 + accuracy4 + accuracy5)/5}%")

### **Evaluating the best model**

In [ ]:
training_losses_new_2 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader2):
        optimizer.zero_grad()

        outputs = model_schmodel(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader2)
    training_losses_new_2.append(epoch_loss)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss}")


plt.plot(range(1, num_epochs + 1), training_losses_new_2, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

model_schmodel.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader2:
        outputs = model_schmodel(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_new_2 = correct / total
print(f"Test Accuracy: {accuracy_new_2 * 100:.2f}%")

In [ ]:
training_losses_new_3 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader3):
        optimizer.zero_grad()

        outputs = model_schmodel(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader3)
    training_losses_new_3.append(epoch_loss)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss}")

plt.plot(range(1, num_epochs + 1), training_losses_new_3, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

model_schmodel.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader3:
        outputs = model_schmodel(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_new_3 = correct / total
print(f"Test Accuracy: {accuracy_new_3 * 100:.2f}%")

In [ ]:
training_losses_new_4 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader4):
        optimizer.zero_grad()

        outputs = model_schmodel(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader4)
    training_losses_new_4.append(epoch_loss)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss}")

plt.plot(range(1, num_epochs + 1), training_losses_new_4, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

model_schmodel.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader4:
        outputs = model_schmodel(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_new_4 = correct / total
print(f"Test Accuracy: {accuracy_new_4 * 100:.2f}%")

In [ ]:
training_losses_new_5 = []


num_epochs = 100
for epoch in range(num_epochs):
    model_schmodel.train()
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_loader5):
        optimizer.zero_grad()

        outputs = model_schmodel(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader5)
    training_losses_new_5.append(epoch_loss)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss}")

plt.plot(range(1, num_epochs + 1), training_losses_new_5, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

model_schmodel.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader5:
        outputs = model_schmodel(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_new_5 = correct / total
print(f"Test Accuracy: {accuracy_new_5 * 100:.2f}%")

In [ ]:
plt.plot(range(1, num_epochs + 1), training_losses, label = "Loss for dataset 1")
plt.plot(range(1, num_epochs + 1), training_losses_new_2, label = "Loss for dataset 2")
plt.plot(range(1, num_epochs + 1), training_losses_new_3, label = "Loss for dataset 3")
plt.plot(range(1, num_epochs + 1), training_losses_new_4, label = "Loss for dataset 4")
plt.plot(range(1, num_epochs + 1), training_losses_new_5, label = "Loss for dataset 5")
plt.legend()
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

print(f"Average test accuracy on best model: {(accuracy + accuracy_new_2 + accuracy_new_3 + accuracy_new_4 + accuracy_new_5)/5}%")